# Food Orders: Data Quality, Cleaning, Analysis and ELT

This notebook completes the eight analysis tasks using the supplied CSV. The raw DataFrame remains unchanged; cleaning is performed on a separate copy. Exact duplicate rows are excluded only after inspection, repeated Order IDs are investigated rather than automatically removed, and unknown financial/category values remain missing. Run the cells from top to bottom.

## Task 1 ? Load and understand the dataset

Load the actual project CSV into `raw_df`, make a separate working copy, inspect the records and types, and document the meaning and expected use of every source field.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from extract import SOURCE_COLUMNS
from transform import FINANCIAL_COLUMNS, REVENUE_COMPONENTS, transform_data
from validate import build_data_quality_report, save_reports, validate_transformed

PROJECT_DIR = Path.cwd()
SOURCE_PATH = PROJECT_DIR / "food_orders.csv"
assert SOURCE_PATH.is_file(), f"CSV not found in notebook working directory: {PROJECT_DIR}"
raw_df = pd.read_csv(SOURCE_PATH)
work_df = raw_df.copy()

print(f"Source: {SOURCE_PATH}")
print("First five rows:")
display(raw_df.head())
print("Last five rows:")
display(raw_df.tail())
print("Random sample of five rows:")
display(raw_df.sample(n=5, random_state=42))
print("Shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())
print("Data types:")
display(raw_df.dtypes.rename("data_type").to_frame())
print("DataFrame info:")
raw_df.info()
print("Numeric statistics:")
display(raw_df.describe(include=["number"]).T)
print("Categorical and datetime-as-text statistics:")
display(raw_df.describe(include=["object"]).T)

column_meanings = {
    "Order ID": ("Unique identifier assigned to an order", "Integer or identifier", "Order counts and duplicate investigation"),
    "Customer ID": ("Identifier of the customer who placed the order", "String", "Customer-level grouping"),
    "Restaurant ID": ("Identifier of the restaurant fulfilling the order", "String", "Restaurant order counts"),
    "Order Date and Time": ("Timestamp when the order was placed", "Datetime", "Order date, hour and delivery duration"),
    "Delivery Date and Time": ("Timestamp when the order was delivered", "Datetime", "Delivery duration"),
    "Order Value": ("Recorded value of the food order", "Numeric currency amount", "Order-value totals, mean and median"),
    "Delivery Fee": ("Delivery fee associated with the order", "Numeric currency amount", "Platform revenue calculation"),
    "Payment Method": ("Method used to pay for the order", "Category/string", "Payment-method order counts"),
    "Discounts and Offers": ("Recorded promotion or offer", "Category/string", "Offer analysis; missing means unknown"),
    "Commission Fee": ("Platform commission recorded for the order", "Numeric currency amount", "Platform revenue calculation"),
    "Payment Processing Fee": ("Payment processing cost recorded for the order", "Numeric currency amount", "Platform revenue calculation"),
    "Refunds/Chargebacks": ("Refund or chargeback amount recorded", "Numeric currency amount", "Platform revenue calculation"),
    "Weather Condition": ("Recorded weather at order time", "Category/string", "Weather frequency and delivery summary"),
    "Order Channel": ("Channel through which the order was submitted", "Category/string", "Channel order counts"),
}
column_info = pd.DataFrame.from_dict(column_meanings, orient="index", columns=["meaning", "expected_data_type", "analytical_use"])
column_info.insert(1, "current_data_type", raw_df.dtypes.astype(str))
column_info.index.name = "column"
display(column_info)
print("Observation: one row represents one recorded food order. Order Value supports order-value analysis; order and delivery timestamps support delivery-time analysis; payment/commission/delivery/processing/refund fields support the specified platform net revenue formula; Payment Method and Order Channel support mix analysis.")

## Task 2 ? Data quality assessment

The next cell answers all nine checks against the actual source: missingness, full duplicates, repeated IDs, types, timestamp parsing/order, suspicious financial/duration values, and category frequencies. An empirical IQR flag is shown for duration outliers; it is descriptive only and is not used as a rejection threshold.

In [ ]:
missing_counts = raw_df.isna().sum().rename("missing_count").to_frame()
missing_counts["missing_percentage"] = (missing_counts["missing_count"] / len(raw_df) * 100).round(2)
print("1?2. Missing values by source column:")
display(missing_counts)

full_duplicate_count = int(raw_df.duplicated(keep="first").sum())
print("3. Fully duplicated rows after the first copy:", full_duplicate_count)

repeated_id_mask = raw_df["Order ID"].duplicated(keep=False)
repeated_id_frequencies = raw_df.loc[repeated_id_mask, "Order ID"].value_counts().sort_index()
print("4. Repeated Order IDs and frequencies:")
display(repeated_id_frequencies.rename("frequency").to_frame())

expected_dtypes = {
    "Order ID": "integer-like identifier", "Customer ID": "string identifier", "Restaurant ID": "string identifier",
    "Order Date and Time": "parseable datetime text", "Delivery Date and Time": "parseable datetime text",
    "Order Value": "numeric", "Delivery Fee": "numeric", "Payment Method": "category/string",
    "Discounts and Offers": "category/string", "Commission Fee": "numeric", "Payment Processing Fee": "numeric",
    "Refunds/Chargebacks": "numeric", "Weather Condition": "category/string", "Order Channel": "category/string",
}
actual_type_checks = []
for column, expected in expected_dtypes.items():
    dtype = raw_df[column].dtype
    numeric_expected = expected in {"integer-like identifier", "numeric"}
    matches = pd.api.types.is_numeric_dtype(dtype) if numeric_expected else not pd.api.types.is_numeric_dtype(dtype)
    actual_type_checks.append({"column": column, "current_type": str(dtype), "expected": expected, "matches_expected": matches})
print("5. Source type checks (dates are expected as parseable text before conversion):")
display(pd.DataFrame(actual_type_checks))

parsed_order = pd.to_datetime(raw_df["Order Date and Time"], errors="coerce")
parsed_delivery = pd.to_datetime(raw_df["Delivery Date and Time"], errors="coerce")
duration = (parsed_delivery - parsed_order).dt.total_seconds() / 60
print("6. Unparseable order timestamps:", int(parsed_order.isna().sum()))
print("6. Unparseable delivery timestamps:", int(parsed_delivery.isna().sum()))
print("7. Delivery-before-order records:", int((duration < 0).sum()))

negative_financial = {column: int((pd.to_numeric(raw_df[column], errors="coerce") < 0).sum()) for column in FINANCIAL_COLUMNS}
print("8. Negative financial values by field:", negative_financial)
print("8. Zero values in financial fields (not automatically treated as invalid):", {column: int((pd.to_numeric(raw_df[column], errors="coerce") == 0).sum()) for column in FINANCIAL_COLUMNS})
q1, q3 = duration.quantile([0.25, 0.75])
iqr = q3 - q1
outlier_mask = (duration < q1 - 1.5 * iqr) | (duration > q3 + 1.5 * iqr)
print("8. Delivery durations:", duration.describe().round(2).to_dict())
print("8. IQR duration outliers (descriptive, not rejected):", int(outlier_mask.sum()))

print("9. Distinct values and frequencies:")
for column in ["Payment Method", "Discounts and Offers", "Weather Condition", "Order Channel"]:
    print(f"\\n{column}")
    display(raw_df[column].value_counts(dropna=False).rename("frequency").to_frame())

print("Verified business-relevant issues: exact duplicate rows exist; several columns contain missing values (including Order Value, fees, payment method and offers); Order IDs repeat and therefore require investigation. Missing refunds/offers are unknown, not zero/no-discount, so indiscriminate imputation would bias revenue and promotion analysis.")

## Task 3 ? Cleaning rules

| Data-quality issue | Handling rule | Reason / business assumption |
|---|---|---|
| Exact duplicate source rows | Keep the first identical row and record later copies as excluded | Identical values across every source field provide no distinct transaction evidence |
| Repeated Order ID | Inspect all rows; remove only if the entire source row is identical; otherwise retain and report | IDs can be reused or repeated in source data and are not by themselves proof of duplicate transactions |
| Missing Order Value | Retain as missing; exclude only from Order Value metrics | A missing order amount cannot be safely reconstructed from other columns |
| Missing Delivery Fee, Commission Fee, Payment Processing Fee or Refunds/Chargebacks | Retain missing; do not impute zero | Zero would invent a financial value and could distort platform revenue |
| Missing Payment Method or Discounts and Offers | Retain as unknown/missing | Missing offer does not prove ?No Discount?; missing payment method is not inferable |
| Invalid/missing timestamps or delivery before order | Reject the record with a reason; never replace duration with zero | Such a duration cannot be trusted for delivery analysis |
| Invalid numeric text or negative financial amounts | Reject and record the reason; retain legitimate zeros | Do not silently coerce invalid or suspicious financial data |
| Unusually long positive durations | Flag descriptively using the observed distribution, but do not reject without a business SLA | No maximum delivery-time rule was provided |

These rules are applied to a separate copy. The source DataFrame and CSV remain unchanged.

## Task 4 ? Clean the dataset

The shared transformation function checks duplicates across the complete source (so duplicates split across 100-row batches are detected), converts timestamps and financial fields safely, retains justified nulls, and records excluded rows with reasons.

In [ ]:
source_with_lineage = raw_df.copy()
source_with_lineage.insert(0, "source_row_number", range(1, len(source_with_lineage) + 1))
cleaned_df, rejected_df, cleaning_details = transform_data(source_with_lineage)
print("First five cleaned rows:")
display(cleaned_df.head())
print("Cleaned data types:")
display(cleaned_df.dtypes.rename("data_type").to_frame())
print("Record accounting:", {key: cleaning_details[key] for key in ["input_records", "exact_duplicate_rows_removed", "accepted_records", "rejected_records"]})
print("Repeated Order IDs before/after cleaning:", cleaning_details["repeated_order_ids_before_cleaning"], cleaning_details["repeated_order_ids_after_cleaning"])
print("Intentionally retained missing values in accepted source columns:")
display(cleaned_df[list(SOURCE_COLUMNS)].isna().sum().loc[lambda values: values.gt(0)].rename("missing_count").to_frame())
print("Rejected/excluded record reasons:")
display(rejected_df["rejection_reason"].value_counts(dropna=False).rename("records").to_frame())

## Task 5 ? Transform the data

The derived duration is the difference between parsed delivery and order timestamps in minutes. `Order_Date` and `Order_Hour` come from the order timestamp. Platform net revenue is computed only if commission, delivery fee, payment-processing fee and refunds/chargebacks are all known. Missing timestamps prevent a trustworthy duration; missing revenue components prevent a complete revenue result. Neither situation is repaired with zero.

In [ ]:
required_revenue_values = cleaned_df[list(REVENUE_COMPONENTS)].notna().all(axis=1)
expected_revenue = (cleaned_df["Commission Fee"] + cleaned_df["Delivery Fee"] - cleaned_df["Payment Processing Fee"] - cleaned_df["Refunds/Chargebacks"])
formula_matches = np.isclose(cleaned_df.loc[required_revenue_values, "Platform_Net_Revenue"].astype(float), expected_revenue.loc[required_revenue_values].astype(float)).all()
print("Derived columns:", ["Delivery_Time_Minutes", "Order_Date", "Order_Hour", "Platform_Net_Revenue"])
print("Negative durations among accepted records:", int(cleaned_df["Delivery_Time_Minutes"].lt(0).sum()))
print("Duration min / median / max (minutes):", cleaned_df["Delivery_Time_Minutes"].min(), cleaned_df["Delivery_Time_Minutes"].median(), cleaned_df["Delivery_Time_Minutes"].max())
print("Revenue formula matches all records with all components:", bool(formula_matches))
print("Records included in platform revenue:", int(cleaned_df["Platform_Net_Revenue"].notna().sum()))
print("Records excluded from revenue because at least one component is missing:", int(cleaned_df["Platform_Net_Revenue"].isna().sum()))
display(cleaned_df[["Order ID", "Order Date and Time", "Delivery Date and Time", "Delivery_Time_Minutes", "Order_Date", "Order_Hour", *REVENUE_COMPONENTS, "Platform_Net_Revenue"]].head())

## Task 6 ? Business questions

All measures below are calculated from the cleaned DataFrame. Financial totals and averages use only rows with a known amount; `count` is shown to make that denominator clear. Platform revenue includes only records with all four revenue components present.

In [ ]:
valid_order_values = cleaned_df["Order Value"].dropna()
known_payment = cleaned_df["Payment Method"].dropna().value_counts()
known_channels = cleaned_df["Order Channel"].dropna().value_counts()
known_weather = cleaned_df["Weather Condition"].dropna().value_counts()
valid_durations = cleaned_df["Delivery_Time_Minutes"].dropna()
revenue_values = cleaned_df["Platform_Net_Revenue"].dropna()
restaurant_counts = cleaned_df["Restaurant ID"].value_counts().head(5)

business_answers = {
    "original_row_count": len(raw_df),
    "cleaned_row_count": len(cleaned_df),
    "records_with_valid_Order_Value": len(valid_order_values),
    "total_Order_Value": valid_order_values.sum(),
    "mean_Order_Value": valid_order_values.mean(),
    "median_Order_Value": valid_order_values.median(),
    "payment_method_most_orders": known_payment.idxmax(),
    "payment_method_order_count": int(known_payment.max()),
    "order_channel_most_orders": known_channels.idxmax(),
    "order_channel_order_count": int(known_channels.max()),
    "most_frequent_weather": known_weather.idxmax(),
    "weather_order_count": int(known_weather.max()),
    "mean_valid_delivery_minutes": valid_durations.mean(),
    "median_valid_delivery_minutes": valid_durations.median(),
    "total_Platform_Net_Revenue": revenue_values.sum(min_count=1),
    "records_in_Platform_Net_Revenue": len(revenue_values),
}
display(pd.Series(business_answers, name="actual_result").to_frame())
print("Five restaurants with the highest order counts:")
display(restaurant_counts.rename("order_count").to_frame())
print(f"Interpretation: order value total/mean/median use {len(valid_order_values)} of {len(cleaned_df)} cleaned rows; missing Order Value is excluded, not imputed. Platform net revenue sums {len(revenue_values)} records with all four components; other records have unknown revenue. Delivery-time statistics use {len(valid_durations)} calculable, accepted records.")
print("Interpretation: the most frequent payment, channel and weather values describe this dataset only; they do not establish why customers chose them.")

## Task 7 ? Visualizations

The charts show order counts by channel and average valid delivery duration by weather. These are descriptive summaries, not evidence that weather causes a delivery-time change.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cleaned_df["Order Channel"].value_counts().sort_values().plot(kind="barh", ax=axes[0], color="#3973ac")
axes[0].set_title("Orders by Order Channel")
axes[0].set_xlabel("Number of orders")
axes[0].set_ylabel("Order channel")
weather_delivery = cleaned_df.groupby("Weather Condition", dropna=False)["Delivery_Time_Minutes"].mean().sort_values()
weather_delivery.plot(kind="bar", ax=axes[1], color="#4c956c")
axes[1].set_title("Average Delivery Time by Weather")
axes[1].set_xlabel("Weather condition")
axes[1].set_ylabel("Average delivery time (minutes)")
axes[1].tick_params(axis="x", rotation=25)
fig.tight_layout()
plt.show()
print("The left chart compares order volume across channels; the right chart compares descriptive average durations among accepted deliveries by weather category.")

## Task 8 ? Validation and saved deliverables

Checks cover the expected schema, datetime/financial types, duplicate treatment and repeated-ID investigation, negative valid durations, manually inspected timestamp arithmetic, the revenue formula, retained missing values, and full record reconciliation. A PASS on missingness means the documented policy was followed, not that every field is non-null.

In [ ]:
validation_report = validate_transformed(cleaned_df, rejected_df, len(raw_df), source_with_lineage, cleaning_details)
print("Validation report:")
display(validation_report)

export_df = cleaned_df.drop(columns=["source_row_number", "load_run_id"], errors="ignore")
cleaned_path = PROJECT_DIR / "food_orders_cleaned.csv"
export_df.to_csv(cleaned_path, index=False)
reloaded_df = pd.read_csv(cleaned_path)
save_reports(source_with_lineage, cleaned_df, rejected_df, validation_report)

expected_export_columns = list(cleaned_df.columns.drop([column for column in ["source_row_number", "load_run_id"] if column in cleaned_df.columns]))
assert list(reloaded_df.columns) == expected_export_columns
assert len(reloaded_df) == len(cleaned_df)
assert validation_report["status"].eq("PASS").all()
print("Cleaned CSV exists:", cleaned_path.is_file())
print("Reloaded shape:", reloaded_df.shape)
print("Export schema matches cleaned source/derived columns:", list(reloaded_df.columns) == expected_export_columns)
print("Validation status:", "PASS" if validation_report["status"].eq("PASS").all() else "FAIL")
print("Reports saved:", ["data_quality_report.csv", "validation_report.csv", "rejected_records.csv"])
print("Final row reconciliation:", len(raw_df), "input =", len(cleaned_df), "accepted +", len(rejected_df), "rejected/excluded")

## Assignment summary

The notebook preserves the raw data, applies documented rules, provides actual Pandas outputs and charts, and saves the cleaned CSV plus quality/validation/rejection reports. The separate MySQL batch pipeline is run from `elt_batch_pipeline.py`; its execution report distinguishes unavailable credentials from successful database work.